# script for investigating met office AWS data

In [5]:
import boto3
import pandas as pd
from io import BytesIO
from botocore import UNSIGNED
from botocore.config import Config

In [15]:
import boto3 # may need to set up aws credentials?
from botocore import UNSIGNED
from botocore.config import Config


BUCKET_NAME = "met-office-land-observations-data"
# ^ most important bit is the bucket name, which is public and can be found in the met office docs

s3 = boto3.client(
    "s3",
    region_name="eu-west-2",
    config=Config(signature_version=UNSIGNED), # no authentication, public. otherwise boto3 expects aws creds
)

response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    MaxKeys=10, # only show 10 items for now
)

for item in response.get("Contents", []):
    print(item["Key"])

202608272359_202608272200_202608272259/altnaharra_no_2_automatic_weather_station_58f8c2e7-962b-47df-bb99-ccdc89e1c607.csv
202608272359_202608272200_202608272259/loch_glascarnoch_automatic_weather_station_ff332a75-9029-47ef-9a95-eecd4e0de270.csv
202608280059_202608272300_202608272359/altnaharra_no_2_automatic_weather_station_715aadb2-7d18-4b42-a56e-2870ea5b8fd6.csv
202608280059_202608272300_202608272359/banagher_caugh_hill_automatic_weather_station_65fe0f62-17aa-4239-83ee-97016fdce484.csv
202608280059_202608272300_202608272359/kielder_castle_automatic_weather_station_4613c790-aa02-4c47-822f-fb6bd8d255c5.csv
202608280059_202608272300_202608272359/kinloss_automatic_weather_station_52e8372e-b915-427b-ba9f-94a9dacd1d51.csv
202608280059_202608272300_202608272359/loch_glascarnoch_automatic_weather_station_cd58957f-438c-4bb3-9169-abddf71e6412.csv
202608280059_202608272300_202608272359/okehampton_east_okement_farm_automatic_weather_station_b470d51e-9376-403b-b1f4-c7ca90d1aab5.csv
202608280059_2

In [16]:
first_key = response["Contents"][0]["Key"]

print(first_key)

202608272359_202608272200_202608272259/altnaharra_no_2_automatic_weather_station_58f8c2e7-962b-47df-bb99-ccdc89e1c607.csv


In [17]:
csv_response = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=first_key,
)

In [18]:
csv_response.keys()

dict_keys(['ResponseMetadata', 'AcceptRanges', 'Expiration', 'LastModified', 'ContentLength', 'ETag', 'ChecksumCRC32', 'ChecksumType', 'VersionId', 'ContentType', 'ServerSideEncryption', 'Metadata', 'ReplicationStatus', 'Body'])

In [19]:
raw_data = csv_response["Body"].read()

print(raw_data[:2000].decode("utf-8"))

timestep|name|longitude|latitude|accumulated_precipitation_1_minute_total|accumulated_precipitation_1_minute_total_qc|air_pressure_near_surface_1_minute_mean|air_pressure_near_surface_1_minute_mean_qc|air_pressure_near_surface_mean_sea_level_1_minute_mean|air_pressure_near_surface_mean_sea_level_1_minute_mean_qc|air_temperature_near_surface_1_minute_mean|air_temperature_near_surface_1_minute_mean_qc|air_temperature_near_surface_secondary_1_minute_mean|air_temperature_near_surface_secondary_1_minute_mean_qc|air_temperature_near_surface_sensor2_1_minute_mean|air_temperature_near_surface_sensor2_1_minute_mean_qc|cloud_base_height_layer1_1_minute_30_minute_rolling_min|cloud_base_height_layer1_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer2_1_minute_30_minute_rolling_min|cloud_base_height_layer2_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer3_1_minute_30_minute_rolling_min|cloud_base_height_layer3_1_minute_30_minute_rolling_min_qc|cloud_cover_layer1_1_minute_30_minute_

In [20]:
import pandas as pd
from io import BytesIO

df = pd.read_csv(BytesIO(raw_data))

In [21]:
df.head()

,timestep|name|longitude|latitude|accumulated_precipitation_1_minute_total|accumulated_precipitation_1_minute_total_qc|air_pressure_near_surface_1_minute_mean|air_pressure_near_surface_1_minute_mean_qc|air_pressure_near_surface_mean_sea_level_1_minute_mean|air_pressure_near_surface_mean_sea_level_1_minute_mean_qc|air_temperature_near_surface_1_minute_mean|air_temperature_near_surface_1_minute_mean_qc|air_temperature_near_surface_secondary_1_minute_mean|air_temperature_near_surface_secondary_1_minute_mean_qc|air_temperature_near_surface_sensor2_1_minute_mean|air_temperature_near_surface_sensor2_1_minute_mean_qc|cloud_base_height_layer1_1_minute_30_minute_rolling_min|cloud_base_height_layer1_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer2_1_minute_30_minute_rolling_min|cloud_base_height_layer2_1_minute_30_minute_rolling_min_qc|cloud_base_height_layer3_1_minute_30_minute_rolling_min|cloud_base_height_layer3_1_minute_30_minute_rolling_min_qc|cloud_cover_layer1_1_minute_30_minute_weighted_mean|cloud_cover_layer1_1_minute_30_minute_weighted_mean_qc|cloud_cover_layer2_1_minute_30_minute_weighted_mean|cloud_cover_layer2_1_minute_30_minute_weighted_mean_qc|cloud_cover_layer3_1_minute_30_minute_weighted_mean|cloud_cover_layer3_1_minute_30_minute_weighted_mean_qc|cloud_cover_total_1_minute_30_minute_weighted_mean|cloud_cover_total_1_minute_30_minute_weighted_mean_qc|dew_point_temperature_1_minute_mean|dew_point_temperature_1_minute_mean_qc|direct_solar_irradiance_1_minute_60_minute_total|direct_solar_irradiance_1_minute_60_minute_total_qc|direct_solar_irradiance_1_minute_binary|direct_solar_irradiance_1_minute_binary_qc|direct_solar_irradiance_1_minute_mean|direct_solar_irradiance_1_minute_mean_qc|direct_solar_irradiance_secondary_1_minute_mean|direct_solar_irradiance_secondary_1_minute_mean_qc|global_radiation_1_minute_mean|global_radiation_1_minute_mean_qc|land_surface_temperature_concrete_1_minute_mean|land_surface_temperature_concrete_1_minute_mean_qc|land_surface_temperature_grass_1_minute_mean|land_surface_temperature_grass_1_minute_mean_qc|meteorological_optical_range_horizontal_1_minute_mean|meteorological_optical_range_horizontal_1_minute_mean_qc|precipitation_intensity_1_minute_rolling_algorithm|precipitation_intensity_1_minute_rolling_algorithm_qc|present_weather_1_minute_10_minute_weighted_mean|present_weather_1_minute_10_minute_weighted_mean_qc|relative_humidity_near_surface_1_minute_mean|relative_humidity_near_surface_1_minute_mean_qc|snow_depth_1_minute_mean|snow_depth_1_minute_mean_qc|soil_temperature_100cm_1_minute_mean|soil_temperature_100cm_1_minute_mean_qc|soil_temperature_10cm_1_minute_mean|soil_temperature_10cm_1_minute_mean_qc|soil_temperature_30cm_1_minute_mean|soil_temperature_30cm_1_minute_mean_qc|wind_direction_near_surface_1_minute_mean|wind_direction_near_surface_1_minute_mean_qc|wind_speed_near_surface_1_minute_mean|wind_speed_near_surface_1_minute_mean_qc
0,2026-08-27T22:46:00Z|Altnaharra No 2 Automatic...
1,2026-08-27T22:47:00Z|Altnaharra No 2 Automatic...
2,2026-08-27T22:48:00Z|Altnaharra No 2 Automatic...
3,2026-08-27T22:49:00Z|Altnaharra No 2 Automatic...
4,2026-08-27T22:50:00Z|Altnaharra No 2 Automatic...


Vertical bar is delimiter, while pd.read.csv assumes commas by default. So, fix this

In [25]:
df = pd.read_csv(BytesIO(raw_data), sep="|")

In [31]:
df.head()

,timestep,name,longitude,latitude,accumulated_precipitation_1_minute_total,accumulated_precipitation_1_minute_total_qc,air_pressure_near_surface_1_minute_mean,air_pressure_near_surface_1_minute_mean_qc,air_pressure_near_surface_mean_sea_level_1_minute_mean,air_pressure_near_surface_mean_sea_level_1_minute_mean_qc,...,soil_temperature_100cm_1_minute_mean,soil_temperature_100cm_1_minute_mean_qc,soil_temperature_10cm_1_minute_mean,soil_temperature_10cm_1_minute_mean_qc,soil_temperature_30cm_1_minute_mean,soil_temperature_30cm_1_minute_mean_qc,wind_direction_near_surface_1_minute_mean,wind_direction_near_surface_1_minute_mean_qc,wind_speed_near_surface_1_minute_mean,wind_speed_near_surface_1_minute_mean_qc
0,2026-08-27T22:46:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN,NaN,NaN,...,13.823007,"{""suspect"": [""2 case(s) of \""missing previous ...",15.559786,"{""suspect"": [""2 case(s) of \""missing previous ...",15.399266,"{""suspect"": [""2 case(s) of \""missing previous ...",NaN,NaN,NaN,NaN
1,2026-08-27T22:47:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN,NaN,NaN,...,13.823007,"{""suspect"": [""2 case(s) of \""missing previous ...",15.554041,"{""suspect"": [""2 case(s) of \""missing previous ...",15.399468,"{""suspect"": [""2 case(s) of \""missing previous ...",NaN,NaN,NaN,NaN
2,2026-08-27T22:48:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN,NaN,NaN,...,13.823107,"{""suspect"": [""2 case(s) of \""missing previous ...",15.543965,"{""suspect"": [""2 case(s) of \""missing previous ...",15.399925,"{""suspect"": [""2 case(s) of \""missing previous ...",NaN,NaN,NaN,NaN
3,2026-08-27T22:49:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN,NaN,NaN,...,13.822957,"{""suspect"": [""1 case(s) of \""qc persistence ch...",15.534491,"{""good"": []}",15.400374,"{""good"": []}",NaN,NaN,NaN,NaN
4,2026-08-27T22:50:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN,NaN,NaN,...,13.823107,"{""suspect"": [""2 case(s) of \""missing previous ...",15.524812,"{""suspect"": [""2 case(s) of \""missing previous ...",15.401033,"{""suspect"": [""2 case(s) of \""missing previous ...",NaN,NaN,NaN,NaN


^ many NaNs

In [26]:
df.shape

(14, 66)

In [27]:
df.columns.tolist()

['timestep',
 'name',
 'longitude',
 'latitude',
 'accumulated_precipitation_1_minute_total',
 'accumulated_precipitation_1_minute_total_qc',
 'air_pressure_near_surface_1_minute_mean',
 'air_pressure_near_surface_1_minute_mean_qc',
 'air_pressure_near_surface_mean_sea_level_1_minute_mean',
 'air_pressure_near_surface_mean_sea_level_1_minute_mean_qc',
 'air_temperature_near_surface_1_minute_mean',
 'air_temperature_near_surface_1_minute_mean_qc',
 'air_temperature_near_surface_secondary_1_minute_mean',
 'air_temperature_near_surface_secondary_1_minute_mean_qc',
 'air_temperature_near_surface_sensor2_1_minute_mean',
 'air_temperature_near_surface_sensor2_1_minute_mean_qc',
 'cloud_base_height_layer1_1_minute_30_minute_rolling_min',
 'cloud_base_height_layer1_1_minute_30_minute_rolling_min_qc',
 'cloud_base_height_layer2_1_minute_30_minute_rolling_min',
 'cloud_base_height_layer2_1_minute_30_minute_rolling_min_qc',
 'cloud_base_height_layer3_1_minute_30_minute_rolling_min',
 'cloud_base_

##### interesting; the file schema contains both the measurements and the qc field (..._qc")

In [28]:
df.dtypes

timestep                                            str
name                                                str
longitude                                       float64
latitude                                        float64
accumulated_precipitation_1_minute_total        float64
                                                 ...   
soil_temperature_30cm_1_minute_mean_qc              str
wind_direction_near_surface_1_minute_mean       float64
wind_direction_near_surface_1_minute_mean_qc    float64
wind_speed_near_surface_1_minute_mean           float64
wind_speed_near_surface_1_minute_mean_qc        float64
Length: 66, dtype: object

In [29]:
df[["timestep", "name", "longitude", "latitude", "global_radiation_1_minute_mean"]].head(10)

,timestep,name,longitude,latitude,global_radiation_1_minute_mean
0,2026-08-27T22:46:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
1,2026-08-27T22:47:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
2,2026-08-27T22:48:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
3,2026-08-27T22:49:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
4,2026-08-27T22:50:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
5,2026-08-27T22:51:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
6,2026-08-27T22:52:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
7,2026-08-27T22:53:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
8,2026-08-27T22:54:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN
9,2026-08-27T22:55:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN


Let's narrow the df to solar vars:


In [32]:
solar_cols = [
    "timestep",
    "name",
    "longitude",
    "latitude",
    "global_radiation_1_minute_mean",
    "global_radiation_1_minute_mean_qc",
    "direct_solar_irradiance_1_minute_mean",
    "direct_solar_irradiance_1_minute_mean_qc",
]

df[solar_cols].head(10)

,timestep,name,longitude,latitude,global_radiation_1_minute_mean,global_radiation_1_minute_mean_qc,direct_solar_irradiance_1_minute_mean,direct_solar_irradiance_1_minute_mean_qc
0,2026-08-27T22:46:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
1,2026-08-27T22:47:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
2,2026-08-27T22:48:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
3,2026-08-27T22:49:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
4,2026-08-27T22:50:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
5,2026-08-27T22:51:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
6,2026-08-27T22:52:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
7,2026-08-27T22:53:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
8,2026-08-27T22:54:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN
9,2026-08-27T22:55:00Z,Altnaharra No 2 Automatic Weather Station,-4.44,58.29,NaN,NaN,NaN,NaN


In [34]:
df["global_radiation_1_minute_mean"].notna().sum()

np.int64(0)

^ this station doesn't have the global radiation data

In [35]:
df.notna().sum().sort_values(ascending=False)

timestep                                            14
name                                                14
longitude                                           14
latitude                                            14
soil_temperature_100cm_1_minute_mean                14
                                                    ..
present_weather_1_minute_10_minute_weighted_mean     0
wind_direction_near_surface_1_minute_mean            0
wind_direction_near_surface_1_minute_mean_qc         0
wind_speed_near_surface_1_minute_mean                0
wind_speed_near_surface_1_minute_mean_qc             0
Length: 66, dtype: int64

Now let's try to answer the more general Q of what stations have this solar data (most recently at least)

In [36]:
batch_prefix = "202608272359_202608272200_202608272259/"

In [37]:
batch_response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    Prefix=batch_prefix,
)

batch_keys = [
    item["Key"]
    for item in batch_response.get("Contents", [])
]

print(f"Found {len(batch_keys)} files")

Found 2 files


In [38]:
from io import BytesIO
import pandas as pd

results = []

for key in batch_keys:
    csv_response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key,
    )

    raw_data = csv_response["Body"].read()

    station_df = pd.read_csv(
        BytesIO(raw_data),
        sep="|",
    )

    results.append({
        "name": station_df["name"].iloc[0],
        "latitude": station_df["latitude"].iloc[0],
        "longitude": station_df["longitude"].iloc[0],
        "global_radiation_count": station_df["global_radiation_1_minute_mean"].notna().sum(),
        "direct_irradiance_count": station_df["direct_solar_irradiance_1_minute_mean"].notna().sum(),
    })

In [39]:
availability = pd.DataFrame(results)

availability.head()

,name,latitude,longitude,global_radiation_count,direct_irradiance_count
0,Altnaharra No 2 Automatic Weather Station,58.29,-4.44,0,0
1,Loch Glascarnoch Automatic Weather Station,57.72,-4.90,0,0


In [40]:
solar_stations = availability[
    (availability["global_radiation_count"] > 0)
    | (availability["direct_irradiance_count"] > 0)
]

solar_stations

,name,latitude,longitude,global_radiation_count,direct_irradiance_count


In [41]:
print(f"Total stations in batch: {len(availability)}")
print(f"Stations with solar data: {len(solar_stations)}")
print(f"Stations with global radiation: {(availability['global_radiation_count'] > 0).sum()}")
print(f"Stations with direct irradiance: {(availability['direct_irradiance_count'] > 0).sum()}")

Total stations in batch: 2
Stations with solar data: 0
Stations with global radiation: 0
Stations with direct irradiance: 0


Didn't need to do all this to figure this out, but: the hardcoded batch_prefix is too limiting; we're not getting all the stations. So let's be more systematic...

In [42]:
response = s3.list_objects_v2(
    Bucket=BUCKET_NAME,
    MaxKeys=1000,
)

keys = [
    item["Key"]
    for item in response.get("Contents", [])
]

print(f"Objects returned: {len(keys)}")
print(f"More objects available: {response['IsTruncated']}")

Objects returned: 1000
More objects available: True


In [ ]:
# now let's see how many different prefixes those first 1000 objects have:
prefixes = [key.split("/")[0] for key in keys]

print(f"Unique prefixes: {len(set(prefixes))}") # set() useful beacuse it removes duplicates, so we can count unique prefixes

Unique prefixes: 99


In [44]:
from collections import Counter

prefix_counts = Counter(prefixes)

prefix_counts.most_common(20)

[('202608291359_202608291200_202608291259', 11),
 ('202608291459_202608291300_202608291359', 11),
 ('202608291559_202608291400_202608291459', 11),
 ('202608291659_202608291500_202608291559', 11),
 ('202608291759_202608291600_202608291659', 11),
 ('202608291859_202608291700_202608291759', 11),
 ('202608291959_202608291800_202608291859', 11),
 ('202608292059_202608291900_202608291959', 11),
 ('202608292159_202608292000_202608292059', 11),
 ('202608292259_202608292100_202608292159', 11),
 ('202608292359_202608292200_202608292259', 11),
 ('202608300059_202608292300_202608292359', 11),
 ('202608300159_202608300000_202608300059', 11),
 ('202608300259_202608300100_202608300159', 11),
 ('202608300359_202608300200_202608300259', 11),
 ('202608300459_202608300300_202608300359', 11),
 ('202608300559_202608300400_202608300459', 11),
 ('202608300659_202608300500_202608300559', 11),
 ('202608300759_202608300600_202608300659', 11),
 ('202608300859_202608300700_202608300759', 11)]

- prefixes look hourly
- each prefix contains only 11 files, so a prefix is likely not the hourly batch for every station. Instead, seems to be grouped accorind to when data were released after processing
- abandon idea of selecting a prefix. instead ask: across all station files available in the bucket, which statations have reported solar data?
- need pagination since S3 only returns 1000 objects per request. boto3 has a paginator

In [46]:
paginator = s3.get_paginator("list_objects_v2")

pages = paginator.paginate(
    Bucket=BUCKET_NAME
)
# ^ not stored in memory, but something python can iterate through, one S3 response page at a time

In [47]:
# now collect all the keys:
all_keys = []

for page in pages:
    for item in page.get("Contents", []):
        all_keys.append(item["Key"])

print(f"Total objects in bucket: {len(all_keys)}")

Total objects in bucket: 54505


In [48]:
filenames = [
    key.split("/")[-1]
    for key in all_keys
]

print(f"Total files: {len(filenames)}")
print(f"Unique filenames: {len(set(filenames))}")

Total files: 54505
Unique filenames: 54505


In [50]:
all_keys[:10]

['202608272359_202608272200_202608272259/altnaharra_no_2_automatic_weather_station_58f8c2e7-962b-47df-bb99-ccdc89e1c607.csv',
 '202608272359_202608272200_202608272259/loch_glascarnoch_automatic_weather_station_ff332a75-9029-47ef-9a95-eecd4e0de270.csv',
 '202608280059_202608272300_202608272359/altnaharra_no_2_automatic_weather_station_715aadb2-7d18-4b42-a56e-2870ea5b8fd6.csv',
 '202608280059_202608272300_202608272359/banagher_caugh_hill_automatic_weather_station_65fe0f62-17aa-4239-83ee-97016fdce484.csv',
 '202608280059_202608272300_202608272359/kielder_castle_automatic_weather_station_4613c790-aa02-4c47-822f-fb6bd8d255c5.csv',
 '202608280059_202608272300_202608272359/kinloss_automatic_weather_station_52e8372e-b915-427b-ba9f-94a9dacd1d51.csv',
 '202608280059_202608272300_202608272359/loch_glascarnoch_automatic_weather_station_cd58957f-438c-4bb3-9169-abddf71e6412.csv',
 '202608280059_202608272300_202608272359/okehampton_east_okement_farm_automatic_weather_station_b470d51e-9376-403b-b1f4-c

In [51]:
all_keys[-10:]

['202609111259_202609111100_202609111159/wight_needles_old_battery_automatic_weather_station_e42bf524-7c5f-403a-b402-04e81ee9544f.csv',
 '202609111259_202609111100_202609111159/wight_st_catherines_point_automatic_weather_station_ebb73029-ad85-42bc-9696-b0463397242a.csv',
 '202609111259_202609111100_202609111159/winchcombe_sudeley_castle_automatic_weather_station_d1a09324-3af4-4f85-8d5a-0b8fc4a456de.csv',
 '202609111259_202609111100_202609111159/winterbourne_no_2_automatic_weather_station_bffbe67f-f795-447e-953d-0bcdb5b5b42f.csv',
 '202609111259_202609111100_202609111159/wisley_automatic_weather_station_0fcc4361-9a1c-40a9-bb85-a7d7e273eeae.csv',
 '202609111259_202609111100_202609111159/wittering_automatic_weather_station_76fc6dd5-a54f-4be8-856d-d33fe50bece8.csv',
 '202609111259_202609111100_202609111159/woburn_automatic_weather_station_a885e487-4b24-4c65-a857-248b953b64d5.csv',
 '202609111259_202609111100_202609111159/writtle_automatic_weather_station_55b33876-c07e-4299-a5d1-86ffd7b0e38

^ i did this at 2:04pm on 2026/09/11 so the latest is at 11:59UTC same day, so basically 12noon UTC or 1pm BST time, so ~65mins latency there for most recent data? but that's the time of release, maybe not the time of data collected...

Now let's take a look at all files on 2026/09/10 (yesterday, given the day I'm writing), at 12noon UTC (note I'm in BST time which is UTC+1)

In [52]:
midday_keys = []

for key in all_keys:
    prefix = key.split("/")[0]
    parts = prefix.split("_")

    if len(parts) == 3:
        observation_start = parts[1]

        if observation_start.startswith("2026091012"):
            midday_keys.append(key)

print(f"Files matching target hour: {len(midday_keys)}")

Files matching target hour: 257


In [53]:
midday_keys[:10]

['202609101359_202609101200_202609101259/aberdaron_automatic_weather_station_2267cd45-4753-4c68-aef4-3027d11e53d5.csv',
 '202609101359_202609101200_202609101259/aberporth_automatic_weather_station_fef339ff-8660-4e09-a33d-47c28a697f08.csv',
 '202609101359_202609101200_202609101259/aboyne_no_2_automatic_weather_station_80ac1a2a-64e1-4c48-9901-3c987cd35df5.csv',
 '202609101359_202609101200_202609101259/achnagart_automatic_weather_station_87717667-4e4e-4a10-ac6a-b566bbdf0486.csv',
 '202609101359_202609101200_202609101259/albemarle_automatic_weather_station_5a3c83c1-60e3-4d9e-bf6e-e4a871b7b3b0.csv',
 '202609101359_202609101200_202609101259/aldergrove_automatic_weather_station_73485184-c1c2-4261-be91-09c1819961da.csv',
 '202609101359_202609101200_202609101259/alice_holt_lodge_automatic_weather_station_338fc409-81dd-4b1d-871f-6a4e00aedfef.csv',
 '202609101359_202609101200_202609101259/almondsbury_automatic_weather_station_b1b4ed4d-d873-4979-85ea-dc6420e7e6db.csv',
 '202609101359_202609101200_

In [54]:
solar_results = []

for key in midday_keys:
    csv_response = s3.get_object(
        Bucket=BUCKET_NAME,
        Key=key,
    )

    raw_data = csv_response["Body"].read()

    station_df = pd.read_csv(
        BytesIO(raw_data),
        sep="|",
    )

    global_count = station_df["global_radiation_1_minute_mean"].notna().sum()
    direct_count = station_df["direct_solar_irradiance_1_minute_mean"].notna().sum()

    solar_results.append({
        "name": station_df["name"].iloc[0],
        "latitude": station_df["latitude"].iloc[0],
        "longitude": station_df["longitude"].iloc[0],
        "global_count": global_count,
        "direct_count": direct_count,
    })

solar_availability = pd.DataFrame(solar_results)

In [58]:
len(solar_availability) # how many stations have data at this hour? (it only needs to have one row of data to be included in the list, even if it has no solar data, i think)

257

In [56]:
(solar_availability["global_count"] > 0).sum() # how many stations have global radiation data at this hour?

np.int64(83)

In [57]:
(solar_availability["direct_count"] > 0).sum()

np.int64(94)

^ answers: 83 have GHI, 94 have DNI. That's not too bad! probably more then the number of solarsense nodes I'd realistically deploy in my dphil...

In [59]:
#which ones?
solar_stations = solar_availability[
    (solar_availability["global_count"] > 0)
    | (solar_availability["direct_count"] > 0)
].copy()

solar_stations.sort_values("name")

,name,latitude,longitude,global_count,direct_count
0,Aberdaron Automatic Weather Station,52.79,-4.74,60,60
1,Aberporth Automatic Weather Station,52.14,-4.57,60,60
5,Aldergrove Automatic Weather Station,54.66,-6.23,60,60
7,Almondsbury Automatic Weather Station,51.55,-2.56,60,60
8,Altnaharra No 2 Automatic Weather Station,58.29,-4.44,60,60
...,...,...,...,...,...
247,Winterbourne No 2 Automatic Weather Station,52.46,-1.93,60,60
248,Wisley Automatic Weather Station,51.31,-0.48,60,60
249,Wittering Automatic Weather Station,52.61,-0.47,60,60
250,Woburn Automatic Weather Station,52.01,-0.60,60,60


In [61]:
import folium

m = folium.Map(
    location=[54.5, -3],
    zoom_start=5,
)

for _, station in solar_stations.iterrows():
    folium.Marker(
        location=[station["latitude"], station["longitude"]],
        tooltip=station["name"],
    ).add_to(m)

m